# Image-to-Image

> Everything to know about image-to-image: the sub-tasks it hides (img2img, inpainting, instruction editing, ControlNet, super-resolution), how conditioning actually works, the mid-2026 model landscape, why there is no single metric, and runnable code to test the leading open models on a 12 GB card.

- skip_showdoc: true
- skip_exec: true

## 1. What is Image-to-Image?

Image-to-image is not one task. It is the Hugging Face pipeline tag for **any model whose input contains an image and whose output is an image**, and the members of that family share an interface, not a mechanism. A super-resolution CNN and an instruction-following 20B diffusion transformer both "do image-to-image" and have nothing else in common. Treat this notebook as an umbrella: pick the sub-task first, then the model.

**Input.** A source image (`PIL.Image`, RGB), plus whatever the sub-task conditions on: a text prompt, an edit instruction, a binary mask, a structural map (edges/depth/pose), a reference image, or nothing at all.

**Output.** A new image, usually at the same resolution as the source (super-resolution being the obvious exception).

**The sub-tasks under the umbrella:**

| Sub-task | Extra conditioning | What it changes | Typical model |
|----------|-------------------|-----------------|---------------|
| img2img / style transfer | text prompt + `strength` | global appearance, loosely anchored to the source layout | SD 1.5 / SDXL / SDXL-Turbo img2img |
| Inpainting | binary mask + prompt | only the masked region | SD inpainting checkpoints, FLUX Fill |
| Outpainting | mask covering new canvas | extends the image beyond its border | same inpainting checkpoints on a padded canvas |
| Instruction editing | natural-language *instruction* | whatever the instruction says, rest preserved | InstructPix2Pix, Qwen-Image-Edit, FLUX.1 Kontext |
| Structural conditioning | edge / depth / pose / scribble map | keeps geometry, regenerates content | ControlNet, T2I-Adapter |
| Subject / style transfer (no finetune) | reference image | imports identity or style from a reference | IP-Adapter |
| Super-resolution / restoration | (degraded image only) | upscales, denoises, deblurs, removes JPEG artifacts | Swin2SR, Real-ESRGAN, SUPIR, OSEDiff |
| Colourisation | (grayscale image only) | adds plausible colour | DDColor, colourisation ControlNets |
| Depth / normal / segmentation maps | none | produces a dense map, not a picture | see `00_Depth_Estimation`, `03_Image_Segmentation` |

That last row is the one people forget: **depth estimation is technically image-to-image** (Depth Anything's HF pipeline tag literally is `image-to-image` on some cards), and dense-map predictors are exactly what supplies ControlNet's conditioning. The tasks feed each other.

**Neighbouring notebooks:** `04_Text_to_Image` (the base generative model everything here conditions), `05_Image_to_Text` (the inverse), `12_Mask_Generation` (SAM produces the masks you inpaint with), `16_Image_Feature_Extraction` (CLIP/DINO embeddings, which is how we score edits in section 4), `18_Video_to_Video` (the same conditioning problem, plus temporal consistency).

---

## 2. Real-World Use Cases

| Use case | Domain | Consumes / produces | Dominant constraint |
|----------|--------|---------------------|---------------------|
| Object removal / generative fill | Consumer photo apps (Google Photos "Magic Eraser", Adobe Photoshop Generative Fill, Samsung Object Eraser) | Photo + brushed mask -> photo with the tourist gone | Seamless boundaries; on-device or sub-second server latency; no hallucinated garbage |
| Product photography at scale | E-commerce (Amazon, Shopify, Etsy ad tooling) | Packshot + background prompt -> lifestyle shot | **Product must be pixel-identical** - a hallucinated logo is a legal problem, not an aesthetic one |
| Concept art and iteration | Games, film, advertising | Sketch/blockout + ControlNet -> rendered frame | Controllability over fidelity; artists need the *composition* they drew, not a nicer one |
| Photo restoration and upscaling | Archives, print, streaming (Topaz, Remini, NVIDIA/AMD upscalers) | Low-res, noisy, JPEG-crushed scan -> clean high-res | No invented detail on faces and text; throughput per image; deterministic output |
| Virtual try-on and staging | Fashion, real estate (IDM-VTON-style pipelines, Zillow-style staging) | Person photo + garment reference -> composited render | Identity and garment fidelity; masking accuracy; latency in an interactive loop |
| Medical image translation | Healthcare research (MRI->CT synthesis, stain normalisation) | Modality A -> modality B | **Structural faithfulness above all** - a generative model that invents a lesion is a catastrophe; hence CycleGAN/UNet still beat diffusion here in practice |
| Satellite / aerial enhancement | Defence, agriculture, mapping | Cloudy or low-res tile -> cleaned tile | Geometric accuracy; auditability; no fabricated features |
| Content moderation and de-identification | Platforms, dashcam / streetview | Frame + face/plate detections -> blurred or synthesised replacement | Recall of the mask (a missed face is a breach); batch cost |

**What the benchmark number hides.** Editing leaderboards score one instruction on one image and average. Production does none of that. The failure that kills a product is almost never "the edit was mediocre" - it is **collateral damage**: you asked for a new background and the model quietly redrew the face, shifted the logo, changed the shade of the brand colour, or subtly deformed the hands. Diffusion editors round-trip the whole image through a VAE, so even the *unedited* pixels come back re-encoded and slightly different; if your pipeline must guarantee the untouched region is bit-identical, you have to composite the source back in outside the mask yourself. Then there is **cost**: a 20B editor is 40 GB of weights and seconds per image, so the shipped system is usually a small distilled model with a big one behind a "make it better" button. And **domain shift** is brutal: models trained on synthetic edit triplets of web photos degrade sharply on documents, screenshots, X-rays, and CAD renders - exactly the images enterprises want edited.

---

## 3. How Modern Image-to-Image Works

This notebook assumes the diffusion fundamentals from `04_Text_to_Image` (latent diffusion, the U-Net/DiT denoiser, the VAE, classifier-free guidance, samplers). What follows is only the **conditioning** story: how you get a source image into a model that was trained to denoise from noise.

**Timeline:**

1. **Paired supervised CNNs (2016-2020).** pix2pix (conditional GAN, needs aligned pairs), CycleGAN (unpaired, cycle-consistency loss), SRCNN/ESRGAN for super-resolution. Still the right answer when you have aligned pairs and need a deterministic, structure-preserving map (medical modality translation, stain normalisation).
2. **SDEdit / img2img (2021-2022).** The trick that started everything: **do not start from pure noise - start from the source**. Encode the image to a latent, add noise up to an intermediate timestep, and denoise from there with a text prompt. Zero training required. It works because a partially noised image still carries the low-frequency layout while the high-frequency identity has been destroyed.
3. **Inpainting checkpoints (2022).** Retrain the U-Net with **9 input channels** (4 noisy latent + 1 downsampled mask + 4 masked-image latent) on random masks, so the model *sees* the hole and the context instead of being blended into it.
4. **ControlNet / T2I-Adapter (Feb 2023).** Add a *structural* conditioning stream (edges, depth, pose, scribble) without touching the base model. This is what made diffusion usable by professionals.
5. **InstructPix2Pix (Jan 2023).** Reformulate editing as instruction following. Generate ~450k synthetic (source, instruction, target) triplets with GPT-3 + Prompt2Prompt, then fine-tune SD with the source latent concatenated to the input. No mask, no per-image prompt engineering.
6. **IP-Adapter (Aug 2023).** ~22M params of decoupled cross-attention that inject a CLIP image embedding as an "image prompt" - subject and style transfer with no fine-tuning.
7. **Unified DiT editors (2024-2026).** The current line. FLUX.1 Kontext (12B, Jun 2025), Step1X-Edit (~19B, Apr 2025, which introduced GEdit-Bench), Qwen-Image-Edit / -2509 / **-2511** (20B MMDiT, Dec 2025), FLUX.2 [dev] (32B + a Mistral-3 24B VLM, Nov 2025), and the closed frontier: Gemini 2.5 Flash Image ("nano-banana"), **Nano Banana Pro / Gemini 3 Pro Image** (Nov 2025), GPT-Image-1. These treat the source image as **extra tokens in the transformer sequence** rather than a channel-concat, which is why they can take several reference images and follow multi-step instructions.

### The `strength` parameter (the most misunderstood knob in img2img)

img2img does **not** run all your denoising steps. It runs

$$t_{\text{start}} = \lfloor \text{num\_inference\_steps} \times \text{strength} \rfloor$$

of them, skipping the earlier (noisier) ones entirely. The source latent is noised to exactly that timestep and denoising resumes from there. So `strength` simultaneously controls:

- **how much of the source survives** - at 0.0 you get the source back untouched, at 1.0 the source is fully destroyed and you are doing plain text-to-image;
- **how long the call takes** - `strength=0.3` with 50 steps runs 15 steps, not 50.

Practical bands: 0.2-0.35 = colour/lighting grade, 0.4-0.6 = restyle with the composition intact (the useful zone), 0.7-0.9 = "inspired by", 1.0 = new image. With SDXL-Turbo (1-4 steps total) you must keep `num_inference_steps * strength >= 1` or the pipeline runs zero steps and hands you back the input.

### Why naive inpainting seams

The tempting approach - run img2img, then paste the generated pixels into the hole - fails for three compounding reasons: the model never saw the mask, so it had no reason to make the fill agree with the context; the VAE is 8x downsampling, so a pixel-space mask edge lands *between* latent cells and bleeds; and the region outside the mask has been re-encoded, so it no longer matches the original even where you did not ask for a change. The fixes, in increasing order of quality:

- **Latent blending.** At every denoising step, overwrite the outside-mask latents with the *correctly noised* source latents for that timestep. This is what `StableDiffusionInpaintPipeline` does when handed an ordinary (non-inpaint) checkpoint. Cheap, and it keeps the context stable, but the model is still guessing about the hole.
- **Inpainting checkpoints** (`stable-diffusion-v1-5/stable-diffusion-inpainting`, FLUX Fill). The 9-channel U-Net above. Trained on random masks, so filling a hole is the *training objective*, not an inference hack. Much better boundary agreement.
- **Feather the mask, and inpaint at full resolution in a crop.** `padding_mask_crop=32` in diffusers crops to the masked region, inpaints at native resolution, and composites back - which fixes the "my 4K photo got downsampled to 512 and the fill is mush" problem.
- **Composite back in pixel space.** After the fact, `Image.composite(generated, source, mask)` so the untouched region is byte-identical to the original. Do this whenever the source is a real photograph you must not alter.

### ControlNet and the zero-convolution

ControlNet does not fine-tune the base model at all. It **clones the encoder** (the U-Net's down-blocks + mid-block), leaves the original frozen, and wires the clone's outputs into the frozen decoder's skip connections through 1x1 convolutions whose **weights and biases are initialised to exactly zero**.

$$\mathbf{y}_c = \mathcal{F}(\mathbf{x};\Theta) + \mathcal{Z}\big(\mathcal{F}(\mathbf{x} + \mathcal{Z}(\mathbf{c};\Theta_{z1});\Theta_c);\Theta_{z2}\big), \qquad \mathcal{Z}(\cdot;\{0,0\}) = 0$$

At training step zero every zero-conv outputs 0, so $\mathbf{y}_c = \mathcal{F}(\mathbf{x};\Theta)$: the network is **bit-identical to the base model**, and no random noise is injected into a 900M-parameter pretrained model by an untrained adapter. The gradient with respect to the zero-conv *weights* is still non-zero (it is proportional to the incoming activations, which are not zero), so the layer learns to open up gradually. That is the whole trick, and it is why ControlNet trains to convergence on ~50k pairs on a single GPU without catastrophic forgetting, and why any SD 1.5 checkpoint can pick up any SD 1.5 ControlNet at inference time. **T2I-Adapter** does the same job with a much smaller (77M) feed-forward adapter that is added once rather than per-block: cheaper and faster, slightly weaker adherence.

### How instruction editors differ

An instruction editor takes the source image as an **extra conditioning stream** and is trained on **edit triplets** (source, instruction, target), so it learns "change only what was asked" as an objective rather than inheriting it from a hack.

- **InstructPix2Pix** concatenates the VAE latent of the source to the noisy latent (8 input channels; the new weights are zero-initialised) and uses **two** guidance scales: `guidance_scale` for text adherence and `image_guidance_scale` for source fidelity. If your edit is being ignored, raise text guidance / lower `image_guidance_scale`; if the image is being destroyed, do the reverse. That two-knob search *is* the fidelity/adherence trade-off made explicit.
- **Kontext / Qwen-Image-Edit / FLUX.2** tokenise the source (VAE tokens, plus VLM tokens in Qwen-Image-Edit's dual-encoding and FLUX.2's Mistral-3 encoder) and append them to the DiT's token sequence - in-context conditioning. That is why they handle multiple reference images, text rendering inside the image, and chained instructions, and why they are 12-32B parameters.

**Cheat sheet:**

| Approach | Training needed | Preserves source | Control | Cost |
|----------|-----------------|------------------|---------|------|
| img2img (SDEdit) | none | via `strength`, loosely | prompt only | cheapest |
| Inpainting checkpoint | pretrained | exactly, outside mask | mask + prompt | cheap |
| ControlNet | +0.36B adapter (once) | structure only, not appearance | edges/depth/pose | +~45% per step |
| T2I-Adapter | +77M adapter (once) | structure only | same, weaker | +~5% |
| IP-Adapter | +22M adapter (once) | subject/style from a *reference* | image prompt | negligible |
| InstructPix2Pix | full fine-tune on triplets | decent, tunable | free-text instruction | cheap (SD 1.5 sized) |
| DiT editor (Kontext/Qwen) | full pretrain, 12-32B | best | free-text, multi-image | does not fit a 12 GB card unquantised |

---

## 4. Evaluation Metrics

**There is no single metric for image-to-image, and anyone who quotes you one number is measuring the wrong thing.** The metric depends entirely on which sub-task you are in, and for editing you need *at least two* numbers that pull against each other.

### Reference-based (restoration, super-resolution: a ground-truth target exists)

**PSNR** - peak signal-to-noise ratio, in dB, over the pixel MSE:

$$\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{\text{MAX}^2}{\text{MSE}}\right), \qquad \text{MSE} = \frac{1}{mn}\sum_{i,j}\big(I(i,j) - K(i,j)\big)^2$$

**SSIM** - structural similarity, computed per local window and averaged:

$$\text{SSIM}(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

with $C_1 = (0.01L)^2$, $C_2 = (0.03L)^2$ and $L$ the dynamic range.

**The pitfall that defines the field: PSNR and SSIM reward blur.** Both are minimised by predicting the conditional *mean* of the plausible outputs, and the mean of many sharp textures is a smooth one. A model that hedges beats a model that commits, so a GAN or diffusion upscaler that produces genuinely convincing texture will often score *worse* on PSNR than a boring L2-trained CNN that a human would reject instantly. This is the **perception-distortion trade-off**, and it is a proven bound, not a tuning failure.

**LPIPS** - the standard fix. Feed both images through a pretrained network (AlexNet/VGG), compare normalised deep activations per layer, learn the channel weights against human 2AFC judgements. Lower is better; it correlates with human preference far better than PSNR. **DISTS** goes further by mixing structure and *texture statistics*, so it stops punishing a resynthesised-but-perceptually-identical texture.

### Distribution-based (no per-image reference)

**FID** compares Inception feature distributions between two image *sets* as a Frechet distance; **KID** is the unbiased kernel version and is the honest choice below ~10k images, where FID's bias is large. Both measure realism of a *set*, never of one image.

### Editing (the hard case: there is no ground truth, and two things must be true at once)

**CLIP directional similarity** asks whether the edit moved the image *in the direction the instruction asked*. Take a caption of the source $c_{src}$ and of the intended result $c_{tgt}$:

$$\text{CLIP}_{\text{dir}} = \cos\big(E_{img}(x_{edit}) - E_{img}(x_{src}),\;\; E_{txt}(c_{tgt}) - E_{txt}(c_{src})\big)$$

It is direction-aware, which plain CLIP score is not: a model that ignores the source entirely and generates a fresh picture of the target caption gets a high *CLIP score* and a poor *directional* score.

**Source preservation** is the counterweight: CLIP or **DINO** image-image cosine similarity between source and edit (DINOv2/DINOv3 embeddings are the better choice for *subject identity* because they are self-supervised on visual structure rather than on captions), or a masked L1/LPIPS over the region that should not have changed.

**The fundamental tension.** These two metrics are trivially gameable *in opposite directions*:

- return the input unchanged -> perfect preservation, zero adherence;
- ignore the input and generate the target caption from scratch -> high adherence, zero preservation.

So **neither number means anything alone**. Report both, and plot them against each other - the 2-D scatter in section 14 is not decoration, it is the only honest summary of an editing model. In 2026 the serious benchmarks all encode this: **MagicBrush** (10k human-annotated triplets, multi-turn), the **Emu Edit test set**, **GEdit-Bench** (from Step1X-Edit; real user instructions, VLM-judged on instruction-following + consistency + quality, now **GEditBench v2** with 23 tasks and an open PVC-Judge), and **ImgEdit-Bench** (NeurIPS 2025 D&B; scores instruction adherence, editing quality, and detail preservation separately). The judge is increasingly a VLM (GPT-5-class or the open PVC-Judge), because no closed-form metric captures "did it change *only* what I asked".

### Speed

Per-image latency at a fixed resolution and step count, and the step count itself - a 4-step SDXL-Turbo edit and a 50-step SD 1.5 edit are not comparable at "seconds per image" without that context. Report VRAM peak too: it decides whether the model ships.

The cell below computes PSNR and SSIM from scratch in numpy (no `skimage` needed) and sketches CLIP directional similarity on fabricated embeddings so the formula is unambiguous.

---

In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter

rng = np.random.default_rng(0)

def psnr(a, b, max_val=1.0):
    "Peak signal-to-noise ratio (dB) between two float images in [0, 1]."
    mse = float(np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2))
    return float("inf") if mse == 0 else 10.0 * np.log10(max_val ** 2 / mse)

def ssim(a, b, max_val=1.0, win=7):
    "Mean SSIM over a uniform sliding window (grayscale float images in [0, 1])."
    C1, C2 = (0.01 * max_val) ** 2, (0.03 * max_val) ** 2
    a, b = a.astype(np.float64), b.astype(np.float64)
    mu_a, mu_b = uniform_filter(a, win), uniform_filter(b, win)
    # E[x^2] - E[x]^2, with the unbiased (N/(N-1)) correction skimage also applies
    n = win ** 2
    cov_norm = n / (n - 1)
    var_a = cov_norm * (uniform_filter(a * a, win) - mu_a ** 2)
    var_b = cov_norm * (uniform_filter(b * b, win) - mu_b ** 2)
    cov_ab = cov_norm * (uniform_filter(a * b, win) - mu_a * mu_b)
    num = (2 * mu_a * mu_b + C1) * (2 * cov_ab + C2)
    den = (mu_a ** 2 + mu_b ** 2 + C1) * (var_a + var_b + C2)
    pad = (win - 1) // 2  # drop the border where the window hangs off the image
    return float(np.mean((num / den)[pad:-pad, pad:-pad]))

# A fabricated "clean" image: a smooth gradient plus a textured patch.
yy, xx = np.mgrid[0:128, 0:128] / 127.0
clean = 0.5 + 0.4 * np.sin(6 * np.pi * xx) * np.cos(4 * np.pi * yy)
clean[40:90, 40:90] += 0.25 * rng.standard_normal((50, 50))  # fine texture
clean = np.clip(clean, 0, 1)

print(f"{'degradation':22s} {'PSNR (dB)':>10s} {'SSIM':>8s}")
for sigma in [0.0, 0.02, 0.05, 0.10, 0.20]:
    noisy = np.clip(clean + sigma * rng.standard_normal(clean.shape), 0, 1)
    print(f"{'noise sigma=' + str(sigma):22s} {psnr(clean, noisy):10.2f} {ssim(clean, noisy):8.4f}")

# The perception-distortion trap: blurring destroys the texture a human would want
# back, yet it scores BETTER than moderate noise on both metrics.
blurred = uniform_filter(clean, 5)
print(f"{'5x5 box blur':22s} {psnr(clean, blurred):10.2f} {ssim(clean, blurred):8.4f}")
print("-> blur beats noise on PSNR/SSIM while looking much worse. This is why LPIPS/DISTS exist.")

# ---------------------------------------------------------------------------
# CLIP directional similarity, on fabricated 8-d embeddings so the algebra is visible.
# In section 14 the same function runs on real CLIP embeddings.
def cos(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-8))

def clip_directional(e_img_src, e_img_edit, e_txt_src, e_txt_tgt):
    "cos( image-space edit direction , text-space edit direction ). Higher = the edit went the way it was asked."
    return cos(e_img_edit - e_img_src, e_txt_tgt - e_txt_src)

e_img_src = rng.standard_normal(8)                    # source image embedding
e_txt_src = rng.standard_normal(8)                    # "a photo of two cats on a couch"
delta_txt = rng.standard_normal(8)                    # the "...in the snow" direction
e_txt_tgt = e_txt_src + delta_txt

candidates = {
    "did nothing (returns source)":     e_img_src.copy(),
    "moved as instructed":              e_img_src + 0.9 * delta_txt,
    "regenerated from scratch":         e_txt_tgt + 0.6 * rng.standard_normal(8),
    "edited the wrong thing":           e_img_src - 0.9 * delta_txt,
}
for name, e_img_edit in candidates.items():
    d = clip_directional(e_img_src, e_img_edit, e_txt_src, e_txt_tgt)
    preserve = cos(e_img_src, e_img_edit)  # source preservation (image-image cosine)
    print(f"{name:32s} CLIP_dir {d:+.3f}   preservation {preserve:+.3f}")

# "did nothing" is undefined/degenerate on direction (zero vector) but perfect on
# preservation; "regenerated from scratch" can score well on direction and badly on
# preservation. Only the PAIR of numbers is informative.

## 5. Datasets

Editing has no equivalent of ImageNet: the supervision is *triplets* (source, instruction, target), and aligned triplets barely exist in nature, so the field runs on synthetic ones.

| Dataset | Contents | Size | Scope | License | Typical use |
|---------|----------|------|-------|---------|-------------|
| [timbrooks/instructpix2pix-clip-filtered](https://huggingface.co/datasets/timbrooks/instructpix2pix-clip-filtered) | Synthetic edit triplets: GPT-3 wrote the instructions, Prompt2Prompt + SD generated the before/after pair, CLIP filtered the junk | ~313k (of ~450k generated) | English instructions, web-photo domain | CC-BY-4.0 (see card) | Train instruction editors. Note the "sources" are themselves generated - so a model trained on it is weaker on real photos |
| [osunlp/MagicBrush](https://huggingface.co/datasets/osunlp/MagicBrush) | **Human-annotated** triplets via DALL-E 2 in the loop; single- and multi-turn, mask-provided and mask-free | ~10k triplets | English, COCO images | CC-BY-4.0 | The standard *fine-tuning + eval* set; fixes InstructPix2Pix's synthetic-source bias |
| [facebook/emu_edit_test_set](https://huggingface.co/datasets/facebook/emu_edit_test_set) | Meta's Emu Edit benchmark: 7 edit categories (local, global, style, background, text, remove, add) | ~3.6k test examples | English | CC-BY-NC (research) | Eval only; the categories matter as much as the average |
| [UCSC-VLAA/HQ-Edit](https://huggingface.co/datasets/UCSC-VLAA/HQ-Edit) | High-resolution GPT-4V + DALL-E 3 generated triplets with detailed instructions | ~197k | English | CC-BY-4.0 | Train; higher visual quality than InstructPix2Pix, still synthetic |
| [ImgEdit](https://github.com/PKU-YuanGroup/ImgEdit) (NeurIPS 2025 D&B) | 1.2M curated edit pairs + ImgEdit-Bench (single- and multi-turn) | 1.2M | English | see repo | Current train + eval standard for open editors |
| [eugenesiow/Div2k](https://huggingface.co/datasets/eugenesiow/Div2k) | DIV2K: 2K-resolution photos with bicubic/unknown LR pairs | 800 train / 100 val | photographic | research | The SR training set |
| [eugenesiow/Set5](https://huggingface.co/datasets/eugenesiow/Set5), [Set14](https://huggingface.co/datasets/eugenesiow/Set14), [Urban100](https://huggingface.co/datasets/eugenesiow/Urban100) | Classical SR eval sets (Urban100 = building facades, brutal on aliasing) | 5 / 14 / 100 images | photographic | research | PSNR/SSIM tables in every SR paper |
| [Places2](http://places2.csail.mit.edu/) | Scene photos; the standard **inpainting** benchmark with published mask sets | 10M+ | scenes | research (non-commercial) | Inpainting train + eval |
| [COCO val2017](https://cocodataset.org/) | Everyday photos with captions | 5k val | photographic | CC-BY-4.0 | Source images for edit benchmarks; the sample below is a COCO image |

**This notebook evaluates on a single COCO image with hand-written instructions.** That is a smoke test, not a benchmark - it exists so the code runs in minutes on a 12 GB card. For a real number, run MagicBrush or GEdit-Bench with a VLM judge. Nothing here is gated; Emu Edit is non-commercial, and DIV2K/Places2 carry research-only terms.

---

## 6. The Model Landscape (mid-2026)

The gap between "the best editor" and "the best editor that fits on this card" is now enormous. Everything at the frontier is a 12-32B DiT.

| Model | Params | License | Sub-task | Architecture | Best for |
|-------|--------|---------|----------|--------------|----------|
| [SD 1.5 img2img](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) | 0.86B U-Net (+CLIP, VAE) | CreativeML OpenRAIL-M | img2img, style | latent diffusion U-Net | The baseline, and the only ecosystem with a ControlNet/LoRA/IP-Adapter for everything |
| [SD 1.5 inpainting](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-inpainting) | 0.86B | CreativeML OpenRAIL-M | inpaint / outpaint | 9-channel U-Net | Cheap mask-conditioned fill |
| [SDXL-Turbo](https://huggingface.co/stabilityai/sdxl-turbo) | 2.6B U-Net | Stability non-commercial (research) | img2img | ADD-distilled SDXL | **Interactive** restyling: 1-2 steps, sub-second |
| [InstructPix2Pix](https://huggingface.co/timbrooks/instruct-pix2pix) | 0.86B | MIT | instruction editing | SD 1.5 + source-latent concat | The only instruction editor that comfortably fits 12 GB. Dated but honest |
| [ControlNet v1.1](https://huggingface.co/lllyasviel/control_v11p_sd15_canny) (canny/depth/pose/scribble/...) | +0.36B per adapter | Apache-2.0 (SD base is OpenRAIL) | structural conditioning | frozen U-Net + trainable encoder copy + zero-convs | Keeping *your* composition |
| [T2I-Adapter](https://huggingface.co/TencentARC/t2i-adapter-canny-sdxl-1.0) | +77M | Apache-2.0 | structural conditioning | lightweight feed-forward adapter | Same job, ~5% overhead instead of ~45% |
| [IP-Adapter](https://huggingface.co/h94/IP-Adapter) | +22M | Apache-2.0 | subject / style transfer | decoupled cross-attention on a CLIP image embed | "Make it look like *this* reference", no finetune |
| [FLUX.1 Kontext [dev]](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev) | 12B | FLUX.1 non-commercial | instruction editing | flow-matching DiT, source as context tokens | Best *open* editor that a 24 GB card can run; **~24 GB bf16 - needs 4-bit here** |
| [Qwen-Image-Edit-2511](https://huggingface.co/Qwen/Qwen-Image-Edit-2511) | 20B MMDiT | Apache-2.0 | instruction editing | Qwen2.5-VL (semantic) + VAE (appearance) dual encoding | Open SOTA: bilingual **text editing inside the image**, identity consistency. ~40 GB bf16, ~20 GB fp8 - **does not fit** |
| [Step1X-Edit](https://huggingface.co/stepfun-ai/Step1X-Edit) | ~19B | Apache-2.0 | instruction editing | VLM + DiT; introduced GEdit-Bench | Fully-open editor with a permissive licence. **Does not fit** |
| [FLUX.2 [dev]](https://huggingface.co/black-forest-labs/FLUX.2-dev) | 32B DiT + Mistral-3 24B VLM | FLUX non-commercial | editing + multi-reference | flow matching, up to 10 references, 4 MP | Nov 2025 frontier open weights. **Does not fit** (4-bit + remote text encoder on a 4090 is the published minimum) |
| Nano Banana Pro / **Gemini 3 Pro Image** | closed | API | instruction editing | closed | The frontier (Nov 2025): text rendering, multi-subject identity, real-world grounding. API-only, watermarked |
| GPT-Image-1 / Seedream / SeedEdit | closed | API | instruction editing | closed | Commercial alternatives |
| [Swin2SR](https://huggingface.co/caidas/swin2SR-classical-sr-x2-64) | 12M | Apache-2.0 | super-resolution | SwinV2 transformer | **transformers-native** SR, x2/x4, tiny. Runs anywhere |
| Real-ESRGAN | 16M | BSD-3 | super-resolution | ESRGAN + synthetic degradations | The production default for anime/photo upscaling (vendor runtime) |
| SUPIR / SeeSR / OSEDiff / StableSR | 1-2.5B+ | mixed | real-world SR | diffusion prior | Hallucinates *plausible* detail; OSEDiff is one-step (~39x faster than SeeSR) |

**Leaderboards:** [ImgEdit-Bench](https://github.com/PKU-YuanGroup/ImgEdit) and [GEditBench v2](https://github.com/ZhangqiJiang07/GEditBench_v2) for editing (both VLM-judged), [LMArena's image-edit arena](https://lmarena.ai/) for human preference, and the NTIRE challenges for super-resolution.

**Who wins what.** On raw edit quality: Nano Banana Pro (closed), then Qwen-Image-Edit-2511 and FLUX.2 among open weights. On *licence*: Qwen-Image-Edit and Step1X-Edit are Apache-2.0; the FLUX line is not commercially free. On *speed*: SDXL-Turbo does img2img in one step - nothing else is close, and for the "interactive restyle" use case in section 2 that beats quality. On *fitting a 12 GB card*: only the SD 1.5 / SDXL family, InstructPix2Pix, the adapters, and the SR models - which is exactly the set this notebook makes runnable.

### What this box cannot run (and the honest reason)

The three models that actually matter for editing in mid-2026 - **FLUX.1 Kontext [dev]** (12B), **Qwen-Image-Edit-2511** (20B), **FLUX.2 [dev]** (32B + a 24B VLM text encoder) - do not fit in 12 GB unquantised, and this notebook will not pretend otherwise. Their `diffusers` APIs are one-liners (`FluxKontextPipeline`, `QwenImageEditPlusPipeline`), so the code is not the problem; the weights are.

The route on this box is 4-bit quantisation plus **sequential** CPU offload, which streams one transformer block at a time through the GPU. Expect **tens of seconds to minutes per image** - the bottleneck becomes PCIe, not the card. Left non-runnable deliberately:

```python
# NOT RUN HERE - needs bitsandbytes and ~10-20 minutes for the first image.
# from diffusers import FluxKontextPipeline, BitsAndBytesConfig, FluxTransformer2DModel
# quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
# tr = FluxTransformer2DModel.from_pretrained(
#     "black-forest-labs/FLUX.1-Kontext-dev", subfolder="transformer",
#     quantization_config=quant, torch_dtype=torch.bfloat16)
# pipe = FluxKontextPipeline.from_pretrained(
#     "black-forest-labs/FLUX.1-Kontext-dev", transformer=tr, torch_dtype=torch.bfloat16)
# pipe.enable_sequential_cpu_offload()          # slowest offload mode, smallest footprint
# pipe(image=source, prompt="make it a snowy winter scene", guidance_scale=2.5).images[0]
```

The realistic options for these models on a 12 GB card are: a hosted API (Replicate/fal/Together), an 8-bit GGUF via ComfyUI, or a bigger GPU. The head-to-head in section 14 therefore compares the three editors that *do* fit, and you should read their scores as "what you can run locally today", not "the state of the art".

---

## 7. Setup

Package roles:

- `diffusers` + `torch` + `accelerate` - the img2img / inpaint / InstructPix2Pix / ControlNet pipelines and `enable_model_cpu_offload()`
- `transformers` - Swin2SR (`AutoModelForImageToImage`) and CLIP (the editing metrics)
- `Pillow` + `numpy` + `scipy` - masks, edge maps, and the metric code above
- `pandas` + `pyecharts` - the benchmark table and charts
- `opencv-python` (**optional**) - `cv2.Canny` gives cleaner edge maps than the Sobel fallback below; the notebook runs either way

`diffusers` is the general-purpose Hugging Face library for diffusion models, the same ecosystem as `transformers` - it is the correct dependency here, not a per-model vendor package.

Every pipeline below is loaded in fp16 with `enable_model_cpu_offload()`, which keeps only the module currently executing on the GPU (U-Net, then VAE, then text encoder) and parks the rest in RAM. On a 12 GB card that is the difference between "comfortable" and "OOM at the VAE decode". Do **not** also call `.to("cuda")` on an offloaded pipeline - accelerate owns the placement.

---

In [ ]:
# diffusers is the general-purpose HF library for diffusion; transformers covers Swin2SR + CLIP.
# %pip install -q torch diffusers transformers accelerate safetensors pillow scipy pandas pyecharts

# Optional: cleaner Canny edges for the ControlNet section, and the webcam demo.
# %pip install -q opencv-python

In [ ]:
import gc
import time
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device)

def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:20s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")

def free_memory():
    "Collect garbage and hand freed VRAM back to the CUDA allocator."
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def place(pipe):
    "Put a diffusers pipeline on the GPU with model CPU offload (12 GB-safe), or on CPU."
    if device == "cpu":
        return pipe.to("cpu")
    pipe.enable_model_cpu_offload()   # accelerate owns placement - do NOT also call .to(device)
    pipe.enable_vae_slicing()         # decode the VAE in slices: a few hundred MB less peak
    pipe.set_progress_bar_config(disable=True)
    return pipe

# All downloads go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)
HF_CACHE = str(DATA_DIR / "hf_cache")

In [ ]:
import urllib.request

import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw, ImageFilter

# The COCO cats image - the de-facto sample across these notebooks.
SAMPLE = DATA_DIR / "coco_cats.jpg"
if not SAMPLE.exists():
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/val2017/000000039769.jpg", SAMPLE
    )

# SD 1.5 works at 512x512; keep everything there so latencies are comparable.
source = Image.open(SAMPLE).convert("RGB").resize((512, 512), Image.LANCZOS)

# The edit we will ask every model for, expressed three ways (this distinction matters):
INSTRUCTION = "make it a snowy winter scene"                       # for instruction editors
CAPTION_SRC = "a photo of two cats lying on a pink couch"          # for CLIP directional sim
CAPTION_TGT = "a photo of two cats lying on a pink couch in a snowy winter scene"
# img2img/ControlNet are NOT instruction models: they want a DESCRIPTION of the desired
# result, not a command. Handing them "make it snowy" produces literal nonsense.

def show(*images, size=256, labels=None):
    "Display PIL images side by side (ECharts cannot draw images; PIL is the right tool)."
    ims = [im.convert("RGB").resize((size, size), Image.LANCZOS) for im in images]
    strip = Image.new("RGB", (size * len(ims), size), "white")
    for i, im in enumerate(ims):
        strip.paste(im, (i * size, 0))
    if labels:
        print(" | ".join(f"{i + 1}. {lab}" for i, lab in enumerate(labels)))
    display(strip)
    return strip

# A mask over the left-hand cat: white = repaint, black = keep. Feathered, because a hard
# 1-pixel mask edge is the classic source of an inpainting seam.
mask = Image.new("L", source.size, 0)
ImageDraw.Draw(mask).ellipse([30, 150, 260, 430], fill=255)
mask = mask.filter(ImageFilter.GaussianBlur(6))

show(source, mask, labels=["source (512x512)", "inpaint mask (white = repaint)"])

## 8. SD 1.5 img2img and the `strength` knob

The SDEdit baseline: encode, noise, denoise with a prompt. `stable-diffusion-v1-5/stable-diffusion-v1-5` is the live repo id - the original `runwayml/stable-diffusion-v1-5` was **deleted by Runway in August 2024**, so any tutorial still using it fails at load. The community mirror under the `stable-diffusion-v1-5` org (and `sd-legacy`) is the same weights.

The cell below sweeps `strength` on the same seed so you can see the single most important behaviour in this notebook: the source dissolving as `strength` climbs, and the call getting *slower* at the same time (because steps executed = `int(num_inference_steps * strength)`).

---

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,            # the checker returns black images on false positives
    requires_safety_checker=False,
    cache_dir=HF_CACHE,
)
place(pipe)
vram("sd15 img2img loaded")

outs, labels = [source], ["source"]
for strength in [0.3, 0.5, 0.75]:
    g = torch.Generator("cpu").manual_seed(0)   # same seed -> the only variable is strength
    t0 = time.perf_counter()
    img = pipe(
        prompt=CAPTION_TGT,                     # a DESCRIPTION, not an instruction
        image=source,
        strength=strength,
        guidance_scale=7.5,
        num_inference_steps=40,                 # actual steps run = int(40 * strength)
        generator=g,
    ).images[0]
    dt = time.perf_counter() - t0
    print(f"strength={strength:.2f}  steps_run={int(40 * strength):2d}  {dt:5.1f}s")
    outs.append(img)
    labels.append(f"strength={strength}")

show(*outs, labels=labels)

del pipe
free_memory()
vram("after sd15 img2img")

## 9. SDXL-Turbo img2img (the interactive case)

Adversarial Diffusion Distillation collapses the sampler to **one step**. For img2img that makes the constraint explicit: the pipeline runs `int(num_inference_steps * strength)` steps, so with `num_inference_steps=2, strength=0.5` you get exactly **1** step - and if that product drops below 1 the pipeline runs zero steps and silently hands you back your input. `guidance_scale=0.0` because Turbo was distilled without classifier-free guidance (passing a guidance scale makes it worse *and* twice as slow).

This is the model behind "restyle as you drag the slider" UIs. Licence is Stability's non-commercial research licence - fine for a notebook, check it before shipping.

---

In [ ]:
from diffusers import AutoPipelineForImage2Image

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=dtype,
    variant="fp16" if device != "cpu" else None,
    cache_dir=HF_CACHE,
)
place(pipe)
vram("sdxl-turbo loaded")

# SDXL wants 512x512+ ; Turbo was trained at 512. num_inference_steps * strength >= 1.
g = torch.Generator("cpu").manual_seed(0)
t0 = time.perf_counter()
turbo_out = pipe(
    prompt=CAPTION_TGT,
    image=source,
    strength=0.6,
    num_inference_steps=2,     # 2 * 0.6 -> 1 actual denoising step
    guidance_scale=0.0,        # ADD-distilled: CFG is not just unnecessary, it hurts
    generator=g,
).images[0]
print(f"sdxl-turbo: {time.perf_counter() - t0:.2f}s for 1 denoising step")

show(source, turbo_out, labels=["source", "sdxl-turbo, 1 step"])

del pipe
free_memory()
vram("after sdxl-turbo")

## 10. Inpainting with a mask-conditioned checkpoint

`stable-diffusion-v1-5/stable-diffusion-inpainting` is the 9-channel U-Net described in section 3: noisy latent (4) + downsampled mask (1) + masked-image latent (4). Because the mask is an *input*, the model composes the fill against the visible context instead of being blended into it after the fact.

Two production details in the cell:

- **`padding_mask_crop=32`** crops to the mask (plus padding), inpaints at native resolution, and composites back - the fix for "my 4K photo got squashed to 512".
- **`Image.composite`** at the end forces the outside-mask pixels back to the byte-identical original, undoing the VAE round-trip. If your source is a real photograph in a real product, do this.

Outpainting is the same call: paste the source onto a larger canvas, mask everything that is new, and run.

---

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
    cache_dir=HF_CACHE,
)
place(pipe)
vram("sd15 inpaint loaded")

g = torch.Generator("cpu").manual_seed(0)
t0 = time.perf_counter()
filled = pipe(
    prompt="a golden retriever puppy curled up asleep, photo",
    image=source,
    mask_image=mask,
    num_inference_steps=30,
    guidance_scale=7.5,
    strength=1.0,               # inpainting: fully regenerate INSIDE the mask
    padding_mask_crop=32,       # inpaint the masked crop at full res, then composite back
    generator=g,
).images[0]
print(f"inpaint: {time.perf_counter() - t0:.1f}s")

# Force the untouched region back to the original pixels (kills the VAE round-trip drift).
hard_mask = mask.point(lambda p: 255 if p > 127 else 0)
composited = Image.composite(filled, source, hard_mask)

show(source, mask, filled, composited,
     labels=["source", "mask", "inpainted", "composited back outside mask"])

# Outpainting, same pipeline:
#   canvas = Image.new("RGB", (768, 512), "white"); canvas.paste(source, (128, 0))
#   omask  = Image.new("L", (768, 512), 255); ImageDraw.Draw(omask).rectangle([128, 0, 639, 511], fill=0)
#   pipe(prompt="a wide shot of a living room", image=canvas, mask_image=omask, ...)

del pipe
free_memory()
vram("after inpaint")

## 11. InstructPix2Pix (instruction editing that fits)

`timbrooks/instruct-pix2pix` (MIT, SD 1.5-sized) is the only instruction editor in this notebook that runs unquantised on a 12 GB card. It is 2023 technology and Qwen-Image-Edit-2511 embarrasses it - but it demonstrates the mechanism cleanly, and the two-guidance-scale API *is* the fidelity/adherence trade-off exposed as a dial:

- **`guidance_scale`** (text, ~7.5): how hard to follow the instruction. Too low and nothing happens.
- **`image_guidance_scale`** (~1.5): how hard to stay faithful to the source. Raise it and the edit gets timid; drop it to 1.0 and the model starts regenerating the picture.

No mask, no target caption - just a command. Note it takes the **instruction**, not the description we fed img2img.

---

In [ ]:
from diffusers import StableDiffusionInstructPix2PixPipeline

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
    cache_dir=HF_CACHE,
)
place(pipe)
vram("instruct-pix2pix loaded")

outs, labels = [source], ["source"]
for img_gs in [1.0, 1.5, 2.0]:   # sweep source-fidelity at fixed text guidance
    g = torch.Generator("cpu").manual_seed(0)
    t0 = time.perf_counter()
    img = pipe(
        prompt=INSTRUCTION,           # an INSTRUCTION, not a description
        image=source,
        num_inference_steps=20,
        guidance_scale=7.5,           # instruction adherence
        image_guidance_scale=img_gs,  # source fidelity
        generator=g,
    ).images[0]
    print(f"image_guidance_scale={img_gs:.1f}  {time.perf_counter() - t0:.1f}s")
    outs.append(img)
    labels.append(f"img_gs={img_gs}")

show(*outs, labels=labels)
print("Low image_guidance_scale -> bolder edit, more collateral damage. High -> timid but safe.")

del pipe
free_memory()
vram("after instruct-pix2pix")

## 12. ControlNet: keeping the composition, replacing everything else

The zero-convolution adapter from section 3. We extract a **Canny edge map** from the source and hand it to `lllyasviel/sd-controlnet-canny` (v1.0; `lllyasviel/control_v11p_sd15_canny` is the v1.1 upgrade and the better default for new work) driving a frozen SD 1.5. The output shares the source's *geometry* and nothing else - which is precisely what the concept-art use case in section 2 wants.

`cv2.Canny` is the canonical edge detector, but `opencv-python` may not be installed here, so the cell falls back to a Sobel-magnitude threshold via `scipy.ndimage`. The fallback edges are thicker (no non-maximum suppression), which mostly costs some fine detail - the ControlNet still tracks them. `controlnet_conditioning_scale` (0.0-1.0+) trades adherence to the edges against freedom for the prompt.

Swap the checkpoint for `-depth`, `-openpose`, `-scribble`, `-seg` and the API is identical; the conditioning maps come from the models in `00_Depth_Estimation`, `17_Keypoint_Detection` and `03_Image_Segmentation`.

---

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

def edge_map(pil_img, low=100, high=200):
    "Canny edges via OpenCV if present, else a Sobel-magnitude threshold (scipy)."
    arr = np.array(pil_img.convert("RGB"))
    try:
        import cv2
        edges = cv2.Canny(arr, low, high)
    except ImportError:
        from scipy.ndimage import gaussian_filter, sobel
        gray = gaussian_filter(np.array(pil_img.convert("L"), dtype=np.float32), 1.4)
        mag = np.hypot(sobel(gray, axis=0), sobel(gray, axis=1))
        edges = (mag > np.percentile(mag, 92)).astype(np.uint8) * 255  # ~8% of pixels are edges
    return Image.fromarray(np.stack([edges] * 3, axis=-1))   # ControlNet wants 3 channels

canny = edge_map(source)

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=dtype, cache_dir=HF_CACHE
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
    cache_dir=HF_CACHE,
)
# UniPC gets usable samples in ~20 steps instead of 50.
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
place(pipe)
vram("controlnet loaded")

g = torch.Generator("cpu").manual_seed(0)
t0 = time.perf_counter()
cn_out = pipe(
    prompt="two origami cats on a wooden bench, paper craft, studio lighting",
    negative_prompt="blurry, lowres, deformed",
    image=canny,                          # the CONDITIONING map, not the source photo
    num_inference_steps=20,
    controlnet_conditioning_scale=1.0,    # lower -> looser adherence to the edges
    generator=g,
).images[0]
print(f"controlnet: {time.perf_counter() - t0:.1f}s")

show(source, canny, cn_out, labels=["source", "canny edges", "controlnet output"])

del pipe, controlnet
free_memory()
vram("after controlnet")

## 13. Super-resolution with Swin2SR (transformers-native)

A different sub-task entirely: no prompt, no diffusion, a **deterministic** 12M-parameter SwinV2 transformer that maps a low-res image to a 2x one (`caidas/swin2SR-classical-sr-x2-64`, Apache-2.0, loaded through `transformers`' `AutoModelForImageToImage`). It is the right tool when you must not invent detail - and the wrong one when you *want* invented detail, which is what SUPIR/SeeSR/OSEDiff's diffusion priors are for.

We downscale the source 4x to fabricate a low-res input, upscale it 2x, and score PSNR/SSIM against a bicubic baseline at the same size. Watch the trap from section 4 play out: the metrics are close, and the *perceptual* gap is much larger than the dB gap suggests.

Run it in fp32 (12M params - the memory is free, and Swin window attention is happier in fp32), on a crop, because self-attention over a full 512x512 image is quadratic and needlessly slow on 4 cores.

---

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageToImage

hr = source.crop((128, 128, 384, 384))                       # 256x256 ground truth
lr = hr.resize((128, 128), Image.BICUBIC)                    # the degraded input (2x down)
bicubic = lr.resize((256, 256), Image.BICUBIC)               # the baseline to beat

sr_id = "caidas/swin2SR-classical-sr-x2-64"
processor = AutoImageProcessor.from_pretrained(sr_id, cache_dir=HF_CACHE)
sr_model = AutoModelForImageToImage.from_pretrained(sr_id, cache_dir=HF_CACHE).to(device).eval()

inputs = processor(lr, return_tensors="pt").to(device)
t0 = time.perf_counter()
with torch.inference_mode():
    out = sr_model(**inputs).reconstruction         # (1, 3, H*2, W*2), float in [0, 1]
print(f"swin2sr: {time.perf_counter() - t0:.2f}s  {tuple(inputs['pixel_values'].shape[-2:])} -> {tuple(out.shape[-2:])}")

arr = out.squeeze(0).float().clamp(0, 1).cpu().numpy().transpose(1, 2, 0)
sr = Image.fromarray((arr * 255).round().astype(np.uint8)).resize(hr.size, Image.BICUBIC)

# Score both against the ground truth, on the luma channel (the SR convention).
def luma(im):
    return np.asarray(im.convert("L"), dtype=np.float64) / 255.0

for name, im in [("bicubic x2", bicubic), ("swin2sr x2", sr)]:
    print(f"{name:12s} PSNR {psnr(luma(hr), luma(im)):5.2f} dB   SSIM {ssim(luma(hr), luma(im)):.4f}")

show(lr, bicubic, sr, hr, labels=["low-res input", "bicubic", "swin2sr", "ground truth"])

del sr_model, processor, inputs, out
free_memory()
vram("after swin2sr")

## 14. Head-to-head Benchmark

Same source image, same intended edit, three editors that fit on the card, loaded and freed **one at a time**:

| Model | How the edit is expressed |
|-------|---------------------------|
| `instruct-pix2pix` | the **instruction** ("make it a snowy winter scene") |
| `sd15 img2img` (strength 0.5) | the **target caption** (img2img cannot follow commands) |
| `sdxl-turbo img2img` (1 step) | the **target caption** |

Two metrics, because - per section 4 - **either one alone is meaningless**:

- **Instruction adherence** = CLIP directional similarity between the image-space edit vector and the text-space caption vector. Higher = the edit went where it was asked.
- **Source preservation** = CLIP image-image cosine between source and edit. Higher = less collateral damage.

The 2-D scatter of those two axes is the honest summary: the top-right corner is a good editor, the bottom-right is a model that did nothing, and the top-left is a model that threw your photo away and generated a new one. Plus a wall-clock bar.

Hardware: RTX 3060 12 GB (fp16, model CPU offload), 512x512, **n = 1 image**. That is a smoke test, not a leaderboard - a real number needs MagicBrush or GEdit-Bench with a VLM judge over thousands of examples, and the ranking below can flip on a different image.

---

In [ ]:
# Pass 1: generate the edits, freeing each pipeline before loading the next.
edits, timings = {}, {}

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix", torch_dtype=dtype, safety_checker=None,
    requires_safety_checker=False, cache_dir=HF_CACHE)
place(pipe)
t0 = time.perf_counter()
edits["instruct-pix2pix"] = pipe(
    prompt=INSTRUCTION, image=source, num_inference_steps=20,
    guidance_scale=7.5, image_guidance_scale=1.5,
    generator=torch.Generator("cpu").manual_seed(0)).images[0]
timings["instruct-pix2pix"] = time.perf_counter() - t0
del pipe; free_memory(); vram("after ip2p")

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=dtype, safety_checker=None,
    requires_safety_checker=False, cache_dir=HF_CACHE)
place(pipe)
t0 = time.perf_counter()
edits["sd15-img2img-s0.5"] = pipe(
    prompt=CAPTION_TGT, image=source, strength=0.5, num_inference_steps=40,
    guidance_scale=7.5, generator=torch.Generator("cpu").manual_seed(0)).images[0]
timings["sd15-img2img-s0.5"] = time.perf_counter() - t0
del pipe; free_memory(); vram("after sd15")

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=dtype,
    variant="fp16" if device != "cpu" else None, cache_dir=HF_CACHE)
place(pipe)
t0 = time.perf_counter()
edits["sdxl-turbo-1step"] = pipe(
    prompt=CAPTION_TGT, image=source, strength=0.6, num_inference_steps=2,
    guidance_scale=0.0, generator=torch.Generator("cpu").manual_seed(0)).images[0]
timings["sdxl-turbo-1step"] = time.perf_counter() - t0
del pipe; free_memory(); vram("after turbo")

show(source, *edits.values(), labels=["source", *edits.keys()])

In [ ]:
# Pass 2: score the (small) PIL outputs with CLIP. The pipelines are already gone,
# so only one model is live at a time - CLIP ViT-B/32 is ~150M params, trivial here.
from transformers import CLIPModel, CLIPProcessor

clip_id = "openai/clip-vit-base-patch32"
clip = CLIPModel.from_pretrained(clip_id, cache_dir=HF_CACHE).to(device).eval()
clip_proc = CLIPProcessor.from_pretrained(clip_id, cache_dir=HF_CACHE)

@torch.inference_mode()
def img_embed(pil):
    x = clip_proc(images=pil, return_tensors="pt").to(device)
    return clip.get_image_features(**x)[0].float().cpu().numpy()

@torch.inference_mode()
def txt_embed(text):
    x = clip_proc(text=[text], return_tensors="pt", padding=True, truncation=True).to(device)
    return clip.get_text_features(**x)[0].float().cpu().numpy()

e_src = img_embed(source)
t_src, t_tgt = txt_embed(CAPTION_SRC), txt_embed(CAPTION_TGT)

rows = []
for name, img in edits.items():
    e_edit = img_embed(img)
    rows.append({
        "model": name,
        "adherence": round(clip_directional(e_src, e_edit, t_src, t_tgt), 4),  # sec-4 helper
        "preservation": round(cos(e_src, e_edit), 4),
        "seconds": round(timings[name], 2),
    })

# A no-op "editor" as the control: it pins the bottom-right corner of the scatter.
rows.append({"model": "identity (no edit)", "adherence": 0.0,
             "preservation": 1.0, "seconds": 0.0})

del clip, clip_proc
free_memory()
vram("after clip")

import pandas as pd
df = pd.DataFrame(rows).sort_values("adherence", ascending=False).reset_index(drop=True)
df

In [ ]:
from pyecharts import options as opts
from pyecharts.charts import Scatter

# The trade-off plot: x = source preservation, y = instruction adherence.
# top-right = good edit | bottom-right = did nothing | top-left = regenerated the image
xs = [float(r["preservation"]) for r in rows]
scatter = Scatter()
scatter.add_xaxis(xaxis_data=[round(x, 4) for x in xs])
for i, r in enumerate(rows):
    ys = [None] * len(rows)
    ys[i] = round(float(r["adherence"]), 4)
    scatter.add_yaxis(
        series_name=r["model"],
        y_axis=ys,
        symbol_size=18,
        label_opts=opts.LabelOpts(is_show=False),
    )
scatter.set_global_opts(
    title_opts=opts.TitleOpts(
        title="Editing trade-off: adherence vs preservation",
        subtitle="n=1 image, RTX 3060, CLIP ViT-B/32. Neither axis means anything alone.",
    ),
    xaxis_opts=opts.AxisOpts(type_="value", name="source preservation (CLIP img-img cos)",
                             min_=0.5, max_=1.02, splitline_opts=opts.SplitLineOpts(is_show=True)),
    yaxis_opts=opts.AxisOpts(type_="value", name="instruction adherence (CLIP directional)",
                             splitline_opts=opts.SplitLineOpts(is_show=True)),
    tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{a}: ({c})"),
    legend_opts=opts.LegendOpts(pos_top="8%"),
)
scatter.render_notebook()

In [ ]:
from pyecharts.charts import Bar

bar = (
    Bar()
    .add_xaxis([r["model"] for r in rows if r["seconds"] > 0])
    .add_yaxis("seconds / image", [r["seconds"] for r in rows if r["seconds"] > 0])
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Latency (512x512, fp16, model CPU offload)"),
        xaxis_opts=opts.AxisOpts(name="model", axislabel_opts=opts.LabelOpts(rotate=20)),
        yaxis_opts=opts.AxisOpts(name="seconds"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
    )
)
bar.render_notebook()

## 15. Interactive Demo: restyle a webcam frame

SDXL-Turbo at one step is fast enough to sit inside an interactive loop - grab a frame, restyle it, show it. This is the "drag the slider" product from section 2 in ten lines.

Guarded end to end: no `opencv-python`, no camera, or a headless container all skip cleanly, and the pipeline is freed either way. Swap `cv2.VideoCapture(0)` for a video file to batch-restyle frames (see `18_Video_to_Video` for why doing this frame-by-frame flickers, and what fixes it).

---

In [ ]:
frame = None
try:
    import cv2  # optional: %pip install -q opencv-python

    cam = cv2.VideoCapture(0)
    ok, bgr = cam.read()
    cam.release()
    if not ok:
        raise RuntimeError("no frame from the camera")
    frame = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)).resize((512, 512))
except Exception as e:  # ImportError (no cv2) / RuntimeError (headless, no camera)
    print(f"webcam unavailable, falling back to the sample image: {type(e).__name__}: {e}")
    frame = source

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=dtype,
    variant="fp16" if device != "cpu" else None, cache_dir=HF_CACHE)
place(pipe)

t0 = time.perf_counter()
styled = pipe(
    prompt="a watercolour painting, soft washes, visible paper texture",
    image=frame, strength=0.6, num_inference_steps=2, guidance_scale=0.0,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]
print(f"one interactive restyle: {time.perf_counter() - t0:.2f}s")
show(frame, styled, labels=["frame", "restyled"])

del pipe
free_memory()
vram("final")

## 16. Going Further

- **Fine-tuning an editor.** The diffusers [InstructPix2Pix training script](https://github.com/huggingface/diffusers/tree/main/examples/instruct_pix2pix) fine-tunes on your own triplets; MagicBrush is the standard set for making a synthetic-trained editor work on real photos. For the DiT editors, LoRA on FLUX.1 Kontext or Qwen-Image-Edit is the practical path (Qwen-Image-Edit-2511 ships with integrated LoRA support) - a few hundred triplets teach a specific edit (a brand style, a product transform) far better than prompt engineering.
- **Training a ControlNet.** [`train_controlnet.py`](https://github.com/huggingface/diffusers/tree/main/examples/controlnet) - ~50k conditioning/image pairs and a single GPU is genuinely enough, thanks to the zero-convs. Build the conditioning maps with the models from `00_Depth_Estimation` / `17_Keypoint_Detection`.
- **Better masks.** Hand-drawn ellipses are a demo. In production the mask comes from SAM/SAM 2 (`12_Mask_Generation`) or from a grounded detector (`13_Zero_Shot_Object_Detection`) - "click the cat, remove the cat" is a two-model pipeline.
- **IP-Adapter for subject/style** without any training: `pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")`, then `pipe(..., ip_adapter_image=reference)`. 22M params, works with every pipeline above (including inpainting and ControlNet, stacked).
- **Faster production runtimes (external, optional).** `torch.compile` on the U-Net/DiT, [stable-fast](https://github.com/chengzeyi/stable-fast), TensorRT, or ONNX Runtime; LCM/Turbo/Lightning distillations for step count; `diffusers`' `enable_sequential_cpu_offload()` when you are truly out of VRAM.
- **Models that still need a vendor runtime.** Real-ESRGAN and SUPIR (super-resolution) and most colourisation models ship their own repos rather than transformers/diffusers integrations - fine as an external service, not as an import in a notebook like this.
- **Related notebooks.** `04_Text_to_Image` (the base models), `12_Mask_Generation` (masks), `00_Depth_Estimation` (ControlNet conditioning), `16_Image_Feature_Extraction` (the CLIP/DINO embeddings behind the metrics), `07_Image_to_Video` and `18_Video_to_Video` (the same conditioning problem in time).

**References**

- [SDEdit: Guided Image Synthesis and Editing with Stochastic Differential Equations](https://arxiv.org/abs/2108.01073) - the img2img/`strength` mechanism
- [Adding Conditional Control to Text-to-Image Diffusion Models (ControlNet, Zhang et al. 2023)](https://arxiv.org/abs/2302.05543) - the zero-convolution
- [InstructPix2Pix: Learning to Follow Image Editing Instructions](https://arxiv.org/abs/2211.09800)
- [IP-Adapter](https://arxiv.org/abs/2308.06721) and [T2I-Adapter](https://arxiv.org/abs/2302.08453)
- [FLUX.1 Kontext](https://bfl.ai/announcements/flux-1-kontext) and [FLUX.2](https://bfl.ai/blog/flux-2)
- [Qwen-Image-Edit-2511 model card](https://huggingface.co/Qwen/Qwen-Image-Edit-2511) and [Qwen-Image technical report](https://arxiv.org/abs/2508.02324)
- [Step1X-Edit (introduced GEdit-Bench)](https://arxiv.org/abs/2504.17761) and [GEditBench v2](https://github.com/ZhangqiJiang07/GEditBench_v2)
- [ImgEdit: A Unified Image Editing Dataset and Benchmark (NeurIPS 2025)](https://huggingface.co/papers/2505.20275)
- [MagicBrush (NeurIPS 2023)](https://arxiv.org/abs/2306.10012)
- [LPIPS: The Unreasonable Effectiveness of Deep Features](https://arxiv.org/abs/1801.03924) and [DISTS](https://arxiv.org/abs/2004.07728)
- [The Perception-Distortion Tradeoff (Blau & Michaeli, 2018)](https://arxiv.org/abs/1711.06077) - why PSNR rewards blur
- [Swin2SR](https://arxiv.org/abs/2209.11345), [OSEDiff (one-step diffusion SR)](https://arxiv.org/abs/2406.08177)
- [diffusers: evaluating diffusion models](https://huggingface.co/docs/diffusers/conceptual/evaluation) - the CLIP directional similarity recipe
- [Nano Banana Pro / Gemini 3 Pro Image](https://blog.google/innovation-and-ai/products/nano-banana-pro/)

---